# Çözücüler ⚙️

Bu egzersizde, farklı `çözücülerin` `LogisticRegression` modelleri üzerindeki etkilerini araştıracaksınız.

👇 Veri kümesini içe aktarmak için aşağıdaki kodu çalıştırın

In [1]:
import pandas as pd

df = pd.read_csv("https://d32aokrjazspmn.cloudfront.net/materials/solvers_dataset.csv")
df.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,sulphates,alcohol,quality rating
0,9.47,5.97,7.36,10.17,6.84,9.15,9.78,9.52,10.34,8.80,6
1,10.05,8.84,9.76,8.38,10.15,6.91,9.70,9.01,9.23,8.80,7
2,10.59,10.71,10.84,10.97,9.03,10.42,11.46,11.25,11.34,9.06,4
3,11.00,8.44,8.32,9.65,7.87,10.92,6.97,11.07,10.66,8.89,8
4,12.12,13.44,10.35,9.95,11.09,9.38,10.22,9.04,7.68,11.38,3


- Veri kümesi farklı şaraplardan oluşmaktadır 🍷
- Özellikler şarapların farklı niteliklerini tanımlar 
- Hedef 🎯 bir uzman tarafından verilen kalite değerlendirmesidir

## 1. Hedef mühendisliği

Bu bölümde, değerlendirmeleri ikili bir hedefe dönüştüreceksiniz.

👇 Her değerlendirme için kaç gözlem bulunmaktadır?

In [2]:
import numpy as np
import time
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.model_selection import cross_validate

target = "quality rating"
df[target].value_counts().sort_index()

quality rating
1     10090
2     10030
3      9838
4      9928
5     10124
6      9961
7      9954
8      9977
9      9955
10    10143
Name: count, dtype: int64

❓ Hedefi ikili sınıflandırma görevine dönüştürerek `y` oluşturun, burada 6'nın altındaki kalite değerlendirmeleri kötü [0], 6 ve üzeri değerlendirmeler iyi [1] olacak

In [3]:
y = (df[target] >= 6).astype(int)
y.head()

0    1
1    1
2    0
3    1
4    0
Name: quality rating, dtype: int64

❓ Yeni ikili hedefin sınıf dengesini kontrol edin

In [4]:
y.value_counts(normalize=True).round(3)

quality rating
0    0.5
1    0.5
Name: proportion, dtype: float64

❓ Özellikleri normalleştirerek `X`'inizi oluşturun. Bu farklı çözücülerin adil karşılaştırılmasına olanak sağlayacaktır.

In [5]:
features = df.drop(columns=target)

scaler = StandardScaler()
X = pd.DataFrame(scaler.fit_transform(features), columns=features.columns)

In [6]:
X.describe().round(2)

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,sulphates,alcohol
count,100000.00,100000.00,100000.00,100000.00,100000.00,100000.00,100000.00,100000.00,100000.00,100000.00
mean,-0.00,-0.00,0.00,-0.00,0.00,-0.00,-0.00,0.00,-0.00,0.00
std,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00
min,-5.95,-3.78,-3.89,-3.43,-3.60,-4.27,-4.07,-4.07,-4.75,-4.57
25%,-0.51,-0.71,-0.70,-0.69,-0.68,-0.67,-0.67,-0.67,-0.68,-0.54
50%,0.20,0.13,0.06,-0.08,-0.18,0.00,-0.00,0.00,0.00,-0.23
75%,0.61,0.64,0.73,0.63,0.56,0.67,0.68,0.67,0.67,0.38
max,3.77,4.12,4.23,4.27,5.41,5.13,4.45,4.24,4.38,5.29


## 2. LogisticRegression çözücüleri

❓ Lojistik Regresyon modelleri farklı **çözücüler** kullanılarak optimize edilebilir. Mevcut çözücülerin karşılaştırmasını yapın:
- Uyum süresi - hangi çözücü **en hızlı**?
- Kesinlik - kesinlik puanları **ne kadar farklı**?

Lojistik Regresyon için mevcut çözücüler: `['newton-cg', 'lbfgs', 'liblinear', 'sag', 'saga']`
 
Bu 5 çözücü hakkında daha fazla bilgi için [bu Stack Overflow konusuna](https://stackoverflow.com/questions/38640109/logistic-regression-python-solvers-defintions) göz atın

In [7]:
solvers = ["newton-cg", "lbfgs", "liblinear", "sag", "saga"]
rows = []

for solver in solvers:
    cv = cross_validate(
        LogisticRegression(solver=solver, max_iter=1000), X, y, cv=5, scoring="accuracy"
    )
    rows.append({
        "solver": solver,
        "fit_time (s)": cv["fit_time"].mean(),
        "accuracy": cv["test_score"].mean(),
    })

solver_results = pd.DataFrame(rows).sort_values("fit_time (s)").reset_index(drop=True)
solver_results.round(4)

,solver,fit_time (s),accuracy
0,lbfgs,0.1541,0.8611
1,newton-cg,0.1615,0.8611
2,liblinear,0.2504,0.8611
3,sag,1.6438,0.8611
4,saga,2.0918,0.8611


In [8]:
# YOUR ANSWER
fastest_solver = solver_results.loc[0, "solver"]
most_accurate_solver = solver_results.sort_values("accuracy", ascending=False).iloc[0]["solver"]

print("En hızlı solver       :", fastest_solver)
print("En yüksek accuracy    :", most_accurate_solver)
print("Accuracy farkı (max-min):", round(solver_results["accuracy"].max() - solver_results["accuracy"].min(), 4))

En hızlı solver       : lbfgs
En yüksek accuracy    : sag
Accuracy farkı (max-min): 0.0


<details>
    <summary>ℹ️ Yorumumuz için buraya tıklayın</summary>

Maliyet fonksiyonumuz 5 çözücünün de bulduğu global bir minimuma sahip olacak kadar "kolay" olduğundan, tüm çözücüler benzer kesinlik puanları üretmelidir. Derin Öğrenme'de olduğu gibi çok karmaşık maliyet fonksiyonları için, farklı çözücüler kayıp fonksiyonunun farklı değerlerinde durabilir.

**Şarap veri kümesi**
    
Mevcut veri kümesinde sklearn'in <a href="https://scikit-learn.org/stable/modules/generated/sklearn.inspection.permutation_importance.html">permutation_importance</a> ile özellik önemini kontrol ederseniz, birçok özelliğin neredeyse 0 önemine sahip olduğunu göreceksiniz. Liblinear çözücü, bir defada sadece *bir* yön boyunca hareket eder ve diğerlerini L1 düzenlileştirme ile düzenler (yani, beta değerlerini 0'a ayarlar), bu da birçok özelliğin hedefi tahmin etmede o kadar da önemli olmadığı bir veri kümesi için iyi bir uyum sağlayabilir.

❗️En iyi çözücüyü arama maliyeti vardır. Varsayılanla (`lbfgs`) devam etmek genel olarak en çok zaman tasarrufu sağlayabilir, sklearn başlamak için hangi çözücüyü seçeceğiniz konusunda fikir vermek için bu tabloyu sunar: 

<img src="https://wagon-public-datasets.s3.amazonaws.com/05-Machine-Learning/04-Under-the-Hood/solvers-chart.png" width=700>

</details>

###  🧪 Kodunuzu test edin

In [9]:
from nbresult import ChallengeResult

result = ChallengeResult(
    'solvers',
    fastest_solver=fastest_solver
)
result.write()
print(result.check())


============================= test session starts ==============================
platform linux -- Python 3.12.9, pytest-8.3.4, pluggy-1.5.0 -- /home/mustafa/.pyenv/versions/3.12.9/bin/python3.12
cachedir: .pytest_cache
rootdir: /home/mustafa/code/mengMustafaCan/sprint-14/S16D4-S-data-solvers/tests
plugins: anyio-4.8.0, typeguard-4.4.2
collecting ... collected 1 item

test_solvers.py::TestSolvers::test_fastest_solver PASSED                 [100%]

============================== 1 passed in 0.01s ===============================


💯 You can commit your code:

git add tests/solvers.pickle

git commit -m 'Completed solvers step'

git push origin master



## 3. Stokastik Gradyan İnişi

Lojistik Regresyon modelleri ayrıca Stokastik Gradyan İnişi ile de optimize edilebilir.

❓ **Stokastik Gradyan İnişi** ile optimize edilmiş bir Lojistik Regresyon modelini değerlendirin. Kesinlik puanı ve eğitim süresi 2. bölümde eğitilen modellerin performansı ile nasıl karşılaştırılır?

<details>
<summary>💡 İpucu</summary>

- Takılırsanız, [SGDClassifier belgelerine](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.SGDClassifier.html) bakın!

</details>

In [10]:
sgd_cv = cross_validate(
    SGDClassifier(loss="log_loss", random_state=42), X, y, cv=5, scoring="accuracy"
)

sgd_row = pd.DataFrame([{
    "solver": "SGD (log_loss)",
    "fit_time (s)": sgd_cv["fit_time"].mean(),
    "accuracy": sgd_cv["test_score"].mean(),
}])
pd.concat([solver_results, sgd_row]).sort_values("fit_time (s)").reset_index(drop=True).round(4)

,solver,fit_time (s),accuracy
0,lbfgs,0.1541,0.8611
1,newton-cg,0.1615,0.8611
2,SGD (log_loss),0.1645,0.8601
3,liblinear,0.2504,0.8611
4,sag,1.6438,0.8611
5,saga,2.0918,0.8611


☝️ SGD modeli, benzer performans için en kısa sürelerden birine sahip olmalıdır (hatta `liblinear`'dan bile daha kısa olabilir). Bu, Gradyan İnişinin her dönemini aynı anda 100k satırı belleğe yüklemek yerine tek bir satırda gerçekleştirmenin doğrudan bir etkisidir.

## 4. Tahminler

❓ En iyi modeli (kısa uyum süresi ve yüksek kesinlik ile dengelenen) kullanarak aşağıdaki şarabın ikili kalitesini (0 veya 1) tahmin edin. Şunları kaydedin:
- `predicted_class`
- `predicted_proba_of_class` (yani modeliniz 1 sınıfını tahmin ettiyse, 1'in sınıf olması gerektiğine inanma olasılığı nedir, 0 ile 1 arasında olmalıdır)

In [11]:
new_wine = pd.read_csv('https://d32aokrjazspmn.cloudfront.net/materials/solvers_new_wine.csv')
new_wine

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,sulphates,alcohol
0,9.54,13.5,12.35,8.78,14.72,9.06,9.67,10.15,11.17,12.17


In [12]:
best_model = LogisticRegression(solver=fastest_solver, max_iter=1000).fit(X, y)

wine = new_wine.drop(columns=target, errors="ignore")[features.columns]
wine_scaled = pd.DataFrame(scaler.transform(wine), columns=features.columns)

predicted_class = int(best_model.predict(wine_scaled)[0])
predicted_proba_of_class = float(best_model.predict_proba(wine_scaled)[0][predicted_class])

print("Tahmin edilen sınıf:", predicted_class, "(0 = kötü, 1 = iyi)")
print(f"Bu sınıfın olasılığı: {predicted_proba_of_class:.3f}")

Tahmin edilen sınıf: 0 (0 = kötü, 1 = iyi)
Bu sınıfın olasılığı: 0.969


# 🏁  Kodunuzu kontrol edin ve notebook'unuzu gönderin

In [13]:
from nbresult import ChallengeResult

result = ChallengeResult(
    'new_data_prediction',
    predicted_class=predicted_class,
    predicted_proba_of_class=predicted_proba_of_class
)
result.write()
print(result.check())


============================= test session starts ==============================
platform linux -- Python 3.12.9, pytest-8.3.4, pluggy-1.5.0 -- /home/mustafa/.pyenv/versions/3.12.9/bin/python3.12
cachedir: .pytest_cache
rootdir: /home/mustafa/code/mengMustafaCan/sprint-14/S16D4-S-data-solvers/tests
plugins: anyio-4.8.0, typeguard-4.4.2
collecting ... collected 2 items

test_new_data_prediction.py::TestNewDataPrediction::test_predicted_class PASSED [ 50%]
test_new_data_prediction.py::TestNewDataPrediction::test_predicted_proba PASSED [100%]

============================== 2 passed in 0.01s ===============================


💯 You can commit your code:

git add tests/new_data_prediction.pickle

git commit -m 'Completed new_data_prediction step'

git push origin master

